In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.multiprocessing as mp
import gymnasium as gym

In [11]:
GAMMA = 0.99
N_STEPS = 5
LR = 1e-3
WORKERS = 4
MAX_EP = 200

print("Parameters set successfully!")

Parameters set successfully!


In [12]:
class Net(nn.Module):

    def __init__(self, state_size, action_size):
        super().__init__()

        self.body = nn.Sequential(
            nn.Linear(state_size, 128),
            nn.ReLU()
        )

        self.actor = nn.Linear(128, action_size)
        self.critic = nn.Linear(128, 1)

    def forward(self, x):
        h = self.body(x)

        logits = self.actor(h)
        value = self.critic(h)

        return logits, value

    def act(self, state):

        state = torch.tensor(
            state,
            dtype=torch.float32
        ).unsqueeze(0)

        logits, value = self.forward(state)

        probs = F.softmax(logits, dim=-1)

        dist = torch.distributions.Categorical(probs)

        action = dist.sample()

        log_prob = dist.log_prob(action)

        return action.item(), log_prob, value

In [13]:
env = gym.make("CartPole-v1")

state_size = env.observation_space.shape[0]
action_size = env.action_space.n

model = Net(state_size, action_size)

print("State size:", state_size)
print("Action size:", action_size)
print("Network created successfully!")

env.close()

State size: 4
Action size: 2
Network created successfully!


In [14]:
def worker(wid, global_model, optimizer, counter, queue):

    env = gym.make("CartPole-v1")

    state_size = env.observation_space.shape[0]
    action_size = env.action_space.n

    local_model = Net(state_size, action_size)

    local_model.load_state_dict(
        global_model.state_dict()
    )

    while True:

        with counter.get_lock():

            if counter.value >= MAX_EP:
                break

        state, _ = env.reset()

        done = False
        episode_reward = 0

        log_probs = []
        values = []
        rewards = []

        while not done:

            action, log_prob, value = local_model.act(state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            done = terminated or truncated

            log_probs.append(log_prob)
            values.append(value)
            rewards.append(reward)

            episode_reward += reward

            state = next_state

            if len(rewards) == N_STEPS or done:

                if done:
                    R = 0.0
                else:
                    _, _, next_value = local_model.act(state)
                    R = next_value.item()

                returns = []

                for r in reversed(rewards):

                    R = r + GAMMA * R
                    returns.insert(0, R)

                returns = torch.tensor(
                    returns,
                    dtype=torch.float32
                )

                values_tensor = torch.cat(values).squeeze(-1)

                advantage = returns - values_tensor

                actor_loss = -(
                    torch.stack(log_probs) *
                    advantage.detach()
                ).mean()

                critic_loss = 0.5 * advantage.pow(2).mean()

                loss = actor_loss + critic_loss

                local_model.zero_grad()

                loss.backward()

                for local_param, global_param in zip(
                    local_model.parameters(),
                    global_model.parameters()
                ):

                    if local_param.grad is not None:
                        global_param._grad = local_param.grad

                optimizer.step()

                local_model.load_state_dict(
                    global_model.state_dict()
                )

                log_probs = []
                values = []
                rewards = []

        with counter.get_lock():

            if counter.value < MAX_EP:

                counter.value += 1
                episode_number = counter.value

            else:
                break

        queue.put(
            (wid, episode_number, episode_reward)
        )

    env.close()

In [15]:
def main():

    env = gym.make("CartPole-v1")

    state_size = env.observation_space.shape[0]
    action_size = env.action_space.n

    env.close()

    global_model = Net(
        state_size,
        action_size
    )

    global_model.share_memory()

    optimizer = torch.optim.Adam(
        global_model.parameters(),
        lr=LR
    )

    counter = mp.Value("i", 0)

    queue = mp.Queue()

    processes = []

    for worker_id in range(WORKERS):

        process = mp.Process(
            target=worker,
            args=(
                worker_id,
                global_model,
                optimizer,
                counter,
                queue
            )
        )

        processes.append(process)

    for process in processes:
        process.start()

    completed = 0

    while completed < MAX_EP:

        worker_id, episode, reward = queue.get()

        completed += 1

        if episode % 10 == 0:

            print(
                f"ep={episode} "
                f"worker={worker_id} "
                f"reward={reward}"
            )

    for process in processes:
        process.join()

    print("\nA3C training completed successfully!")

In [16]:
%%writefile a3c.py

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.multiprocessing as mp
import gymnasium as gym

GAMMA = 0.99
N_STEPS = 5
LR = 1e-3
WORKERS = 4
MAX_EP = 190


class Net(nn.Module):

    def __init__(self, state_size, action_size):
        super().__init__()

        self.body = nn.Sequential(
            nn.Linear(state_size, 128),
            nn.ReLU()
        )

        self.actor = nn.Linear(128, action_size)
        self.critic = nn.Linear(128, 1)

    def forward(self, x):
        h = self.body(x)
        return self.actor(h), self.critic(h)

    def act(self, state):

        state = torch.tensor(
            state,
            dtype=torch.float32
        ).unsqueeze(0)

        logits, value = self.forward(state)

        probs = F.softmax(logits, dim=-1)

        dist = torch.distributions.Categorical(probs)

        action = dist.sample()

        return action.item(), dist.log_prob(action), value


def worker(wid, global_model, optimizer, counter, queue):

    env = gym.make("CartPole-v1")

    local_model = Net(
        env.observation_space.shape[0],
        env.action_space.n
    )

    local_model.load_state_dict(
        global_model.state_dict()
    )

    while True:

        with counter.get_lock():
            if counter.value >= MAX_EP:
                break

        state, _ = env.reset()

        done = False
        episode_reward = 0

        log_probs = []
        values = []
        rewards = []

        while not done:

            action, log_prob, value = local_model.act(state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            done = terminated or truncated

            log_probs.append(log_prob)
            values.append(value)
            rewards.append(reward)

            episode_reward += reward

            state = next_state

            if len(rewards) == N_STEPS or done:

                if done:
                    R = 0.0
                else:
                    _, _, next_value = local_model.act(state)
                    R = next_value.item()

                returns = []

                for r in reversed(rewards):
                    R = r + GAMMA * R
                    returns.insert(0, R)

                returns = torch.tensor(
                    returns,
                    dtype=torch.float32
                )

                values_tensor = torch.cat(values).squeeze(-1)

                advantage = returns - values_tensor

                actor_loss = -(
                    torch.stack(log_probs) *
                    advantage.detach()
                ).mean()

                critic_loss = 0.5 * advantage.pow(2).mean()

                loss = actor_loss + critic_loss

                local_model.zero_grad()

                loss.backward()

                for local_param, global_param in zip(
                    local_model.parameters(),
                    global_model.parameters()
                ):

                    if local_param.grad is not None:
                        global_param._grad = local_param.grad

                optimizer.step()

                local_model.load_state_dict(
                    global_model.state_dict()
                )

                log_probs = []
                values = []
                rewards = []

        with counter.get_lock():

            if counter.value < MAX_EP:
                counter.value += 1
                episode_number = counter.value
            else:
                break

        queue.put(
            (wid, episode_number, episode_reward)
        )

    env.close()


def main():

    env = gym.make("CartPole-v1")

    state_size = env.observation_space.shape[0]
    action_size = env.action_space.n

    env.close()

    global_model = Net(
        state_size,
        action_size
    )

    global_model.share_memory()

    optimizer = torch.optim.Adam(
        global_model.parameters(),
        lr=LR
    )

    counter = mp.Value("i", 0)

    queue = mp.Queue()

    processes = []

    for worker_id in range(WORKERS):

        process = mp.Process(
            target=worker,
            args=(
                worker_id,
                global_model,
                optimizer,
                counter,
                queue
            )
        )

        processes.append(process)

    for process in processes:
        process.start()

    completed = 0

    while completed < MAX_EP:

        worker_id, episode, reward = queue.get()

        completed += 1

        if episode % 10 == 0:

            print(
                f"ep={episode} "
                f"worker={worker_id} "
                f"reward={reward}"
            )

    for process in processes:
        process.join()

    print("\nA3C training completed successfully!")


if __name__ == "__main__":

    try:
        mp.set_start_method(
            "spawn",
            force=True
        )
    except RuntimeError:
        pass

    main()

Overwriting a3c.py


In [17]:
!python a3c.py

ep=10 worker=2 reward=13.0
ep=20 worker=3 reward=18.0
ep=30 worker=2 reward=19.0
ep=40 worker=0 reward=10.0
ep=50 worker=2 reward=15.0
ep=60 worker=2 reward=15.0
ep=70 worker=3 reward=21.0
ep=80 worker=3 reward=12.0
ep=90 worker=1 reward=24.0
ep=100 worker=3 reward=26.0
ep=110 worker=2 reward=36.0
ep=120 worker=1 reward=34.0
ep=130 worker=0 reward=36.0
ep=140 worker=2 reward=48.0
ep=150 worker=2 reward=34.0
ep=160 worker=2 reward=147.0
ep=170 worker=3 reward=24.0
ep=180 worker=3 reward=24.0
ep=190 worker=2 reward=25.0

A3C training completed successfully!
